# 16. Multi-Target Knowledge Base Build

This notebook merges the processed outputs from notebooks 09-15 into one RAG-ready knowledge base.

Inputs:
- ChEMBL drug recommendations
- PubMed evidence summaries
- ClinicalTrials.gov evidence summaries
- openFDA label summaries
- DGIdb interaction evidence
- Open Targets disease associations
- UniProt target metadata
- DrugCentral activity and indication evidence

Outputs:
- `data/processed/multi_target_therapy_knowledge_base.csv`
- `data/processed/multi_target_therapy_knowledge_base_chunks.jsonl`
- `data/processed/multi_target_therapy_knowledge_base_summary.csv`

This notebook does not call external APIs. It only merges already processed local files.

In [ ]:
import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project root:", PROJECT_ROOT)
print("Processed folder:", PROCESSED_DIR)

## 1. Load Processed Source Files

The merge is intentionally tolerant: if a source file is missing, the notebook continues and leaves those evidence fields empty. This makes it easier to rerun while you are still building the pipeline.

In [ ]:
def load_csv(filename, required=False):
    path = PROCESSED_DIR / filename
    if not path.exists():
        message = f"Missing: {path}"
        if required:
            raise FileNotFoundError(message)
        print(message)
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f"Loaded {filename}: {df.shape}")
    return df


recommendations_df = load_csv("multi_target_drug_recommendations.csv", required=True)
pubmed_summary_df = load_csv("multi_target_pubmed_summary.csv")
clinical_summary_df = load_csv("multi_target_clinical_trials_summary.csv")
openfda_summary_df = load_csv("multi_target_openfda_summary.csv")
dgidb_interactions_df = load_csv("multi_target_dgidb_interactions.csv")
opentargets_associations_df = load_csv("multi_target_opentargets_associations.csv")
uniprot_metadata_df = load_csv("multi_target_uniprot_target_metadata.csv")
drugcentral_summary_df = load_csv("multi_target_drugcentral_summary.csv")
external_crosscheck_df = load_csv("multi_target_external_crosscheck_summary.csv")

print("Ready to merge.")

## 2. Helper Functions

In [ ]:
def normalize_name(value):
    if pd.isna(value):
        return ""
    return re.sub(r"[^A-Z0-9]+", "", str(value).upper())


def safe_join(values, limit=8):
    cleaned = [str(value).strip() for value in values if pd.notna(value) and str(value).strip()]
    unique_values = list(dict.fromkeys(cleaned))
    return " | ".join(unique_values[:limit])


def clean_text(value, max_chars=None):
    if pd.isna(value):
        return ""
    text = re.sub(r"\s+", " ", str(value)).strip()
    if max_chars is not None and len(text) > max_chars:
        text = text[:max_chars].rstrip() + "..."
    return text


def bool_value(value):
    if pd.isna(value):
        return False
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def numeric(value, default=0):
    try:
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default

## 3. Start with ChEMBL Candidate Drugs

Each row in the final knowledge base represents one target-drug candidate.

In [ ]:
kb_df = recommendations_df.copy()
kb_df["normalised_drug_name"] = kb_df["drug_name"].apply(normalize_name)

base_columns = [
    "target_symbol", "target_display_name", "target_full_name", "target_chembl_id", "target_pref_name",
    "drug_name", "normalised_drug_name", "molecule_chembl_id", "molecule_type", "action_type",
    "mechanism_of_action", "approval_status", "max_phase", "first_approval",
]
for column in base_columns:
    if column not in kb_df.columns:
        kb_df[column] = ""
kb_df = kb_df[base_columns]

print("Initial KB rows:", len(kb_df))
print("Targets:", kb_df["target_symbol"].nunique())
print("Unique drugs:", kb_df["normalised_drug_name"].nunique())
display(kb_df.head())

## 4. Add Target Metadata from UniProt

In [ ]:
if not uniprot_metadata_df.empty:
    uniprot_columns = [
        "target_symbol", "uniprot_accession", "uniprot_id", "protein_name", "gene_names",
        "organism", "function_summary", "subcellular_locations", "disease_notes",
        "ensembl_ids", "chembl_ids", "hgnc_ids", "pdb_count", "go_count", "url",
    ]
    uniprot_merge_df = uniprot_metadata_df[[c for c in uniprot_columns if c in uniprot_metadata_df.columns]].drop_duplicates("target_symbol")
    uniprot_merge_df = uniprot_merge_df.rename(columns={
        "url": "uniprot_url",
        "function_summary": "target_function_summary",
        "disease_notes": "target_disease_notes",
    })
    kb_df = kb_df.merge(uniprot_merge_df, on="target_symbol", how="left")

print("After UniProt merge:", kb_df.shape)
display(kb_df.head())

## 5. Add Drug-Level Evidence Summaries

These joins use `target_symbol`, `drug_name`, and `molecule_chembl_id` where possible.

In [ ]:
merge_keys = ["target_symbol", "drug_name", "molecule_chembl_id"]

if not pubmed_summary_df.empty:
    pubmed_cols = [c for c in ["target_symbol", "drug_name", "molecule_chembl_id", "pubmed_count", "top_pubmed_titles"] if c in pubmed_summary_df.columns]
    kb_df = kb_df.merge(pubmed_summary_df[pubmed_cols], on=merge_keys, how="left")
else:
    kb_df["pubmed_count"] = 0
    kb_df["top_pubmed_titles"] = ""

if not clinical_summary_df.empty:
    clinical_cols = [c for c in [
        "target_symbol", "drug_name", "molecule_chembl_id", "clinical_trial_count",
        "active_trial_count", "late_phase_trial_count", "top_trial_titles", "clinical_evidence_score",
    ] if c in clinical_summary_df.columns]
    kb_df = kb_df.merge(clinical_summary_df[clinical_cols], on=merge_keys, how="left")
else:
    kb_df["clinical_trial_count"] = 0
    kb_df["active_trial_count"] = 0
    kb_df["late_phase_trial_count"] = 0
    kb_df["top_trial_titles"] = ""
    kb_df["clinical_evidence_score"] = 0

if not openfda_summary_df.empty:
    openfda_cols = [c for c in [
        "target_symbol", "drug_name", "molecule_chembl_id", "label_count", "mentions_target_in_label",
        "brand_names", "generic_names", "top_indications", "has_openfda_label",
    ] if c in openfda_summary_df.columns]
    kb_df = kb_df.merge(openfda_summary_df[openfda_cols], on=merge_keys, how="left")
else:
    kb_df["label_count"] = 0
    kb_df["mentions_target_in_label"] = False
    kb_df["brand_names"] = ""
    kb_df["generic_names"] = ""
    kb_df["top_indications"] = ""
    kb_df["has_openfda_label"] = False

print("After drug-level evidence merge:", kb_df.shape)
display(kb_df.head())

## 6. Add DGIdb and DrugCentral Cross-Checks

DGIdb and DrugCentral do not always use ChEMBL molecule IDs, so this step uses normalized drug names within each target.

In [ ]:
if not dgidb_interactions_df.empty:
    dgidb_work_df = dgidb_interactions_df.copy()
    if "normalised_drug_name" not in dgidb_work_df.columns:
        dgidb_work_df["normalised_drug_name"] = dgidb_work_df["drug_name"].apply(normalize_name)
    if "matches_project_drug" in dgidb_work_df.columns:
        dgidb_work_df = dgidb_work_df[dgidb_work_df["matches_project_drug"].apply(bool_value)]

    dgidb_summary_df = dgidb_work_df.groupby(["target_symbol", "normalised_drug_name"], dropna=False).agg(
        dgidb_interaction_count=("drug_name", "size"),
        dgidb_sources=("sources", safe_join),
        dgidb_interaction_types=("interaction_types", safe_join),
        dgidb_max_interaction_score=("interaction_score", "max"),
    ).reset_index()
    kb_df = kb_df.merge(dgidb_summary_df, on=["target_symbol", "normalised_drug_name"], how="left")
else:
    kb_df["dgidb_interaction_count"] = 0
    kb_df["dgidb_sources"] = ""
    kb_df["dgidb_interaction_types"] = ""
    kb_df["dgidb_max_interaction_score"] = 0

if not drugcentral_summary_df.empty:
    drugcentral_work_df = drugcentral_summary_df.copy()
    drugcentral_work_df["normalised_drug_name"] = drugcentral_work_df["drugcentral_name"].apply(normalize_name)
    if "matches_project_drug" in drugcentral_work_df.columns:
        drugcentral_work_df = drugcentral_work_df[drugcentral_work_df["matches_project_drug"].apply(bool_value)]

    drugcentral_merge_df = drugcentral_work_df.groupby(["target_symbol", "normalised_drug_name"], dropna=False).agg(
        drugcentral_activity_count=("activity_count", "sum"),
        drugcentral_min_activity_value=("min_activity_value", "min"),
        drugcentral_activity_types=("activity_types", safe_join),
        drugcentral_activity_sources=("activity_sources", safe_join),
        drugcentral_indication_count=("indication_count", "sum"),
        drugcentral_off_label_count=("off_label_count", "sum"),
        drugcentral_contraindication_count=("contraindication_count", "sum"),
        drugcentral_top_indications=("top_indications", safe_join),
    ).reset_index()
    kb_df = kb_df.merge(drugcentral_merge_df, on=["target_symbol", "normalised_drug_name"], how="left")
else:
    kb_df["drugcentral_activity_count"] = 0
    kb_df["drugcentral_min_activity_value"] = None
    kb_df["drugcentral_activity_types"] = ""
    kb_df["drugcentral_activity_sources"] = ""
    kb_df["drugcentral_indication_count"] = 0
    kb_df["drugcentral_off_label_count"] = 0
    kb_df["drugcentral_contraindication_count"] = 0
    kb_df["drugcentral_top_indications"] = ""

print("After DGIdb/DrugCentral merge:", kb_df.shape)
display(kb_df.head())

## 7. Add Target-Level Open Targets Disease Evidence

In [ ]:
if not opentargets_associations_df.empty:
    opentargets_summary_df = opentargets_associations_df.sort_values(
        ["target_symbol", "association_score"], ascending=[True, False]
    ).groupby("target_symbol", dropna=False).agg(
        opentargets_disease_rows=("disease_name", "size"),
        opentargets_total_disease_count=("total_associated_disease_count", "max"),
        top_opentargets_diseases=("disease_name", safe_join),
        top_opentargets_score=("association_score", "max"),
    ).reset_index()
    kb_df = kb_df.merge(opentargets_summary_df, on="target_symbol", how="left")
elif not external_crosscheck_df.empty:
    external_cols = [c for c in [
        "target_symbol", "opentargets_disease_rows", "opentargets_total_disease_count", "top_opentargets_diseases",
    ] if c in external_crosscheck_df.columns]
    kb_df = kb_df.merge(external_crosscheck_df[external_cols], on="target_symbol", how="left")
else:
    kb_df["opentargets_disease_rows"] = 0
    kb_df["opentargets_total_disease_count"] = 0
    kb_df["top_opentargets_diseases"] = ""
    kb_df["top_opentargets_score"] = 0

print("After Open Targets merge:", kb_df.shape)
display(kb_df.head())

## 8. Clean Missing Values and Compute Evidence Scores

The score is not a medical ranking. It is a simple project-level signal that helps sort candidates with more supporting evidence.

In [ ]:
count_columns = [
    "pubmed_count", "clinical_trial_count", "active_trial_count", "late_phase_trial_count",
    "label_count", "dgidb_interaction_count", "drugcentral_activity_count",
    "drugcentral_indication_count", "drugcentral_off_label_count", "drugcentral_contraindication_count",
    "opentargets_disease_rows", "opentargets_total_disease_count",
]
for column in count_columns:
    if column not in kb_df.columns:
        kb_df[column] = 0
    kb_df[column] = pd.to_numeric(kb_df[column], errors="coerce").fillna(0)

text_columns = [
    "target_function_summary", "target_disease_notes", "top_pubmed_titles", "top_trial_titles",
    "brand_names", "generic_names", "top_indications", "dgidb_sources",
    "dgidb_interaction_types", "drugcentral_activity_types", "drugcentral_activity_sources",
    "drugcentral_top_indications", "top_opentargets_diseases",
]
for column in text_columns:
    if column not in kb_df.columns:
        kb_df[column] = ""
    kb_df[column] = kb_df[column].fillna("").apply(clean_text)

bool_columns = ["mentions_target_in_label", "has_openfda_label"]
for column in bool_columns:
    if column not in kb_df.columns:
        kb_df[column] = False
    kb_df[column] = kb_df[column].apply(bool_value)

if "clinical_evidence_score" not in kb_df.columns:
    kb_df["clinical_evidence_score"] = 0
kb_df["clinical_evidence_score"] = pd.to_numeric(kb_df["clinical_evidence_score"], errors="coerce").fillna(0)

# Evidence components: simple and explainable, not a clinical decision score.
kb_df["has_pubmed_evidence"] = kb_df["pubmed_count"] > 0
kb_df["has_clinical_trial_evidence"] = kb_df["clinical_trial_count"] > 0
kb_df["has_late_phase_trial_evidence"] = kb_df["late_phase_trial_count"] > 0
kb_df["has_fda_label_evidence"] = kb_df["label_count"] > 0
kb_df["has_dgidb_evidence"] = kb_df["dgidb_interaction_count"] > 0
kb_df["has_drugcentral_activity"] = kb_df["drugcentral_activity_count"] > 0
kb_df["has_drugcentral_indication"] = kb_df["drugcentral_indication_count"] > 0
kb_df["has_opentargets_disease_evidence"] = kb_df["opentargets_disease_rows"] > 0
kb_df["is_approved_or_phase4"] = kb_df.apply(
    lambda row: "approved" in str(row.get("approval_status", "")).lower() or numeric(row.get("max_phase")) >= 4,
    axis=1,
)

kb_df["evidence_source_count"] = kb_df[[
    "has_pubmed_evidence", "has_clinical_trial_evidence", "has_fda_label_evidence",
    "has_dgidb_evidence", "has_drugcentral_activity", "has_opentargets_disease_evidence",
]].sum(axis=1)

kb_df["evidence_score"] = (
    kb_df["is_approved_or_phase4"].astype(int) * 2.0
    + kb_df["has_pubmed_evidence"].astype(int) * 1.0
    + kb_df["has_clinical_trial_evidence"].astype(int) * 1.5
    + kb_df["has_late_phase_trial_evidence"].astype(int) * 1.0
    + kb_df["has_fda_label_evidence"].astype(int) * 1.5
    + kb_df["has_dgidb_evidence"].astype(int) * 1.0
    + kb_df["has_drugcentral_activity"].astype(int) * 1.0
    + kb_df["has_drugcentral_indication"].astype(int) * 0.75
    + kb_df["has_opentargets_disease_evidence"].astype(int) * 0.5
)

kb_df["evidence_strength"] = pd.cut(
    kb_df["evidence_score"],
    bins=[-1, 2.5, 5.0, 99],
    labels=["low", "moderate", "strong"],
)

print("Evidence scoring complete.")
display(kb_df[["target_symbol", "drug_name", "approval_status", "evidence_source_count", "evidence_score", "evidence_strength"]].head(20))

## 9. Build Human-Readable Evidence Text

This text becomes the base for RAG chunks.

In [ ]:
def build_evidence_text(row):
    parts = []
    parts.append(f"Target: {row.get('target_display_name') or row.get('target_symbol')} ({row.get('target_symbol')}).")
    if clean_text(row.get("target_full_name")):
        parts.append(f"Target full name: {clean_text(row.get('target_full_name'))}.")
    if clean_text(row.get("protein_name")):
        parts.append(f"UniProt protein name: {clean_text(row.get('protein_name'))}.")
    if clean_text(row.get("target_function_summary")):
        parts.append(f"Target function: {clean_text(row.get('target_function_summary'), 700)}")

    parts.append(f"Drug candidate: {clean_text(row.get('drug_name'))}.")
    if clean_text(row.get("molecule_chembl_id")):
        parts.append(f"ChEMBL molecule ID: {clean_text(row.get('molecule_chembl_id'))}.")
    if clean_text(row.get("action_type")):
        parts.append(f"Action type: {clean_text(row.get('action_type'))}.")
    if clean_text(row.get("mechanism_of_action")):
        parts.append(f"Mechanism: {clean_text(row.get('mechanism_of_action'))}.")
    parts.append(f"Approval status: {clean_text(row.get('approval_status'))}; max phase: {row.get('max_phase')}.")

    parts.append(
        f"Evidence summary: PubMed articles={int(row.get('pubmed_count', 0))}, "
        f"clinical trials={int(row.get('clinical_trial_count', 0))}, "
        f"openFDA labels={int(row.get('label_count', 0))}, "
        f"DGIdb interactions={int(row.get('dgidb_interaction_count', 0))}, "
        f"DrugCentral activities={int(row.get('drugcentral_activity_count', 0))}."
    )

    if clean_text(row.get("top_pubmed_titles")):
        parts.append(f"PubMed evidence: {clean_text(row.get('top_pubmed_titles'), 500)}")
    if clean_text(row.get("top_trial_titles")):
        parts.append(f"Clinical trial evidence: {clean_text(row.get('top_trial_titles'), 500)}")
    if clean_text(row.get("top_indications")):
        parts.append(f"openFDA indications: {clean_text(row.get('top_indications'), 500)}")
    if clean_text(row.get("drugcentral_top_indications")):
        parts.append(f"DrugCentral indications: {clean_text(row.get('drugcentral_top_indications'), 350)}")
    if clean_text(row.get("top_opentargets_diseases")):
        parts.append(f"Target disease associations from Open Targets: {clean_text(row.get('top_opentargets_diseases'), 350)}")

    parts.append(
        f"Project evidence strength: {row.get('evidence_strength')} "
        f"(score={round(float(row.get('evidence_score', 0)), 2)}, sources={int(row.get('evidence_source_count', 0))})."
    )
    return "\n".join(parts)


kb_df["evidence_text"] = kb_df.apply(build_evidence_text, axis=1)

print(kb_df["evidence_text"].iloc[0][:1500])

## 10. Save Final Knowledge Base

In [ ]:
preferred_columns = [
    "target_symbol", "target_display_name", "target_full_name", "target_chembl_id", "uniprot_accession",
    "uniprot_id", "protein_name", "gene_names", "ensembl_ids", "hgnc_ids", "chembl_ids",
    "drug_name", "normalised_drug_name", "molecule_chembl_id", "molecule_type", "action_type",
    "mechanism_of_action", "approval_status", "max_phase", "first_approval",
    "pubmed_count", "top_pubmed_titles", "clinical_trial_count", "active_trial_count",
    "late_phase_trial_count", "top_trial_titles", "clinical_evidence_score",
    "label_count", "has_openfda_label", "mentions_target_in_label", "brand_names", "generic_names", "top_indications",
    "dgidb_interaction_count", "dgidb_sources", "dgidb_interaction_types", "dgidb_max_interaction_score",
    "drugcentral_activity_count", "drugcentral_min_activity_value", "drugcentral_activity_types",
    "drugcentral_activity_sources", "drugcentral_indication_count", "drugcentral_off_label_count",
    "drugcentral_contraindication_count", "drugcentral_top_indications",
    "opentargets_disease_rows", "opentargets_total_disease_count", "top_opentargets_diseases",
    "target_function_summary", "target_disease_notes", "subcellular_locations", "pdb_count", "go_count", "uniprot_url",
    "has_pubmed_evidence", "has_clinical_trial_evidence", "has_late_phase_trial_evidence",
    "has_fda_label_evidence", "has_dgidb_evidence", "has_drugcentral_activity",
    "has_drugcentral_indication", "has_opentargets_disease_evidence", "is_approved_or_phase4",
    "evidence_source_count", "evidence_score", "evidence_strength", "evidence_text",
]

for column in preferred_columns:
    if column not in kb_df.columns:
        kb_df[column] = ""

kb_final_df = kb_df[preferred_columns].sort_values(
    ["target_symbol", "evidence_score", "max_phase", "drug_name"],
    ascending=[True, False, False, True],
).reset_index(drop=True)

knowledge_base_file = PROCESSED_DIR / "multi_target_therapy_knowledge_base.csv"
kb_final_df.to_csv(knowledge_base_file, index=False)

print("Saved:", knowledge_base_file)
print("Rows:", len(kb_final_df))
print("Targets:", kb_final_df["target_symbol"].nunique())
display(kb_final_df.head(20))

## 11. Build RAG Chunks

Each chunk represents one target-drug candidate. These chunks can be indexed into ChromaDB or another vector database in the next stage.

In [ ]:
chunks = []

for index, row in kb_final_df.iterrows():
    target_symbol = row["target_symbol"]
    molecule_id = row.get("molecule_chembl_id") or "unknown_molecule"
    drug_slug = normalize_name(row.get("drug_name")) or f"row_{index}"
    doc_id = f"multi_target::{target_symbol}::{molecule_id}::{drug_slug}"

    chunk = {
        "id": doc_id,
        "text": row["evidence_text"],
        "metadata": {
            "target_symbol": target_symbol,
            "target_display_name": row.get("target_display_name", ""),
            "drug_name": row.get("drug_name", ""),
            "molecule_chembl_id": row.get("molecule_chembl_id", ""),
            "approval_status": row.get("approval_status", ""),
            "max_phase": numeric(row.get("max_phase")),
            "evidence_strength": str(row.get("evidence_strength", "")),
            "evidence_score": numeric(row.get("evidence_score")),
            "evidence_source_count": int(numeric(row.get("evidence_source_count"))),
            "has_pubmed_evidence": bool(row.get("has_pubmed_evidence")),
            "has_clinical_trial_evidence": bool(row.get("has_clinical_trial_evidence")),
            "has_fda_label_evidence": bool(row.get("has_fda_label_evidence")),
            "has_dgidb_evidence": bool(row.get("has_dgidb_evidence")),
            "has_drugcentral_activity": bool(row.get("has_drugcentral_activity")),
            "has_opentargets_disease_evidence": bool(row.get("has_opentargets_disease_evidence")),
            "source_table": "multi_target_therapy_knowledge_base.csv",
        },
    }
    chunks.append(chunk)

chunks_file = PROCESSED_DIR / "multi_target_therapy_knowledge_base_chunks.jsonl"
with chunks_file.open("w") as f:
    for chunk in chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

print("Saved:", chunks_file)
print("Chunks:", len(chunks))
print(json.dumps(chunks[0], indent=2)[:1800])

## 12. Build Summary Table

This gives a quick health check of the knowledge base by target.

In [ ]:
summary_df = kb_final_df.groupby("target_symbol").agg(
    candidate_drug_count=("drug_name", "nunique"),
    approved_or_phase4_count=("is_approved_or_phase4", "sum"),
    strong_evidence_count=("evidence_strength", lambda s: int((s.astype(str) == "strong").sum())),
    moderate_evidence_count=("evidence_strength", lambda s: int((s.astype(str) == "moderate").sum())),
    drugs_with_pubmed=("has_pubmed_evidence", "sum"),
    drugs_with_trials=("has_clinical_trial_evidence", "sum"),
    drugs_with_fda_labels=("has_fda_label_evidence", "sum"),
    drugs_with_dgidb=("has_dgidb_evidence", "sum"),
    drugs_with_drugcentral=("has_drugcentral_activity", "sum"),
    average_evidence_score=("evidence_score", "mean"),
).reset_index()

summary_df["average_evidence_score"] = summary_df["average_evidence_score"].round(2)

summary_file = PROCESSED_DIR / "multi_target_therapy_knowledge_base_summary.csv"
summary_df.to_csv(summary_file, index=False)

print("Saved:", summary_file)
display(summary_df)

## 13. Final Check

If this looks correct, the next project stage is vector indexing and retrieval evaluation.

In [ ]:
print("Multi-Target Knowledge Base Build Complete")
print("=" * 70)
print("Knowledge base rows:", len(kb_final_df))
print("Targets:", kb_final_df["target_symbol"].nunique())
print("Chunks:", len(chunks))
print("Files created:")
print("-", knowledge_base_file)
print("-", chunks_file)
print("-", summary_file)

print("\nTop candidates by evidence score:")
display(kb_final_df[[
    "target_symbol", "drug_name", "approval_status", "max_phase", "evidence_source_count",
    "evidence_score", "evidence_strength",
]].head(30))

print("\nSummary by target:")
display(summary_df)
